In [10]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    matthews_corrcoef, cohen_kappa_score, roc_auc_score,
    average_precision_score, log_loss
)

SEED = 42
CSV_PATH = "Synthetic_Financial_datasets_log.csv"

TARGET_COL = "isFraud"
GROUP_COL  = "nameOrig"          # leakage-free split (customer-based)

# Hard leakage/ID columns to drop from features
DROP_COLS = ["isFlaggedFraud", "nameDest", "newbalanceOrig", "newbalanceDest"]  # nameOrig kept only for grouping then removed

TEST_SIZE = 0.20

# Optional: for quicker experiments on laptop
# Set to None for full dataset
MAX_ROWS = None   # e.g., 2_000_000


In [11]:
# --- Use only required columns (prevents RAM spikes) ---
usecols = [
    "step", "type", "amount",
    "nameOrig", "oldbalanceOrg", "newbalanceOrig",
    "nameDest", "oldbalanceDest", "newbalanceDest",
    "isFraud", "isFlaggedFraud"
]

# Memory-friendly dtypes
dtype_map = {
    "step": "int16",
    "type": "category",
    "amount": "float32",
    "nameOrig": "string",
    "nameDest": "string",
    "oldbalanceOrg": "float32",
    "newbalanceOrig": "float32",
    "oldbalanceDest": "float32",
    "newbalanceDest": "float32",
    "isFraud": "int8",
    "isFlaggedFraud": "int8",
}

df = pd.read_csv(CSV_PATH, usecols=usecols, dtype=dtype_map, nrows=MAX_ROWS)

# Basic sanity
df = df.dropna(subset=[TARGET_COL]).copy()
df[TARGET_COL] = df[TARGET_COL].astype("int8")

print("Loaded shape:", df.shape)
print("Fraud rate:", float(df[TARGET_COL].mean()))
print(df["type"].value_counts())


Loaded shape: (6362620, 11)
Fraud rate: 0.001290820448180152
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [12]:
# ============================================================
# Leakage-free, index-safe GROUP split cell (PaySim)
# Fixes: "IndexError: positional indexers are out-of-bounds"
# ============================================================

import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# ---- Safety: make sure df index is clean and aligned
df = df.reset_index(drop=True).copy()

# ---- Validate required columns exist
required_cols = [TARGET_COL, GROUP_COL]
missing_req = [c for c in required_cols if c not in df.columns]
if missing_req:
    raise ValueError(f"Missing required columns in df: {missing_req}")

# ---- Drop rows with missing target or missing group (rare, but safe)
df = df.dropna(subset=[TARGET_COL, GROUP_COL]).reset_index(drop=True)

# ---- Create target & groups from SAME df (aligned)
y = df[TARGET_COL].astype(np.int8).values
groups_all = df[GROUP_COL].astype(str).values

# ---- Build feature columns (keep GROUP_COL in X for now, drop later)
drop_set = set(DROP_COLS) if "DROP_COLS" in globals() else set()
feature_cols = [c for c in df.columns if c not in drop_set and c != TARGET_COL]

# ---- Build X from SAME df (aligned)
X_raw = df.loc[:, feature_cols].copy()

# ---- FINAL alignment checks (prevents out-of-bounds)
nX, ny, ng = len(X_raw), len(y), len(groups_all)
print(f"[Check] lengths -> X:{nX}, y:{ny}, groups:{ng}")
if not (nX == ny == ng):
    raise ValueError("Misalignment detected: X_raw, y, and groups_all lengths differ.")

# ---- Leakage-free split: users/groups in train won't appear in test
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
tr_idx, te_idx = next(gss.split(X_raw, y, groups=groups_all))

# ---- Extra sanity: indices must be within bounds
if tr_idx.max() >= nX or te_idx.max() >= nX:
    raise ValueError("Split indices are out of bounds. Check df alignment earlier in pipeline.")

# ---- Create train/test sets
X_train_raw = X_raw.iloc[tr_idx].reset_index(drop=True)
X_test_raw  = X_raw.iloc[te_idx].reset_index(drop=True)
y_train = y[tr_idx]
y_test  = y[te_idx]

groups_train = groups_all[tr_idx]
groups_test  = groups_all[te_idx]

print("Train shape:", X_train_raw.shape, " | Fraud rate:", float(y_train.mean()))
print("Test  shape:", X_test_raw.shape,  " | Fraud rate:", float(y_test.mean()))

# ---- Drop group/identity column from features (VERY important)
X_train_raw = X_train_raw.drop(columns=[GROUP_COL], errors="ignore")
X_test_raw  = X_test_raw.drop(columns=[GROUP_COL], errors="ignore")

print("[OK] Split done. GROUP_COL removed from features, groups_train kept for grouped OOF.")


[Check] lengths -> X:6362620, y:6362620, groups:6362620
Train shape: (5090094, 6)  | Fraud rate: 0.0012905459113328752
Test  shape: (1272526, 6)  | Fraud rate: 0.001291918593411844
[OK] Split done. GROUP_COL removed from features, groups_train kept for grouped OOF.


In [19]:
import numpy as np
import pandas as pd

def add_safe_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    eps = np.float32(1e-3)  # Increased from 1e-6 to prevent infinity values

    # Required column groups
    req_orig = ["oldbalanceOrg", "newbalanceOrig", "amount"]
    req_dest = ["oldbalanceDest", "newbalanceDest", "amount"]

    # --- Orig delta + error + ratio
    if all(c in X.columns for c in ["newbalanceOrig", "oldbalanceOrg"]):
        X["orig_balance_delta"] = (X["newbalanceOrig"] - X["oldbalanceOrg"]).astype(np.float32)
    else:
        missing = [c for c in ["newbalanceOrig", "oldbalanceOrg"] if c not in X.columns]
        print("[Warn] Skipping orig_balance_delta, missing:", missing)

    if all(c in X.columns for c in ["amount", "oldbalanceOrg"]):
        X["amount_to_oldbalanceOrg"] = (X["amount"] / (X["oldbalanceOrg"] + eps)).astype(np.float32)
    else:
        missing = [c for c in ["amount", "oldbalanceOrg"] if c not in X.columns]
        print("[Warn] Skipping amount_to_oldbalanceOrg, missing:", missing)

    if all(c in X.columns for c in ["oldbalanceOrg", "amount", "newbalanceOrig"]):
        X["orig_error"] = (X["oldbalanceOrg"] - X["amount"] - X["newbalanceOrig"]).astype(np.float32)
    else:
        missing = [c for c in ["oldbalanceOrg", "amount", "newbalanceOrig"] if c not in X.columns]
        print("[Warn] Skipping orig_error, missing:", missing)

    # --- Dest delta + error + ratio
    if all(c in X.columns for c in ["newbalanceDest", "oldbalanceDest"]):
        X["dest_balance_delta"] = (X["newbalanceDest"] - X["oldbalanceDest"]).astype(np.float32)
    else:
        missing = [c for c in ["newbalanceDest", "oldbalanceDest"] if c not in X.columns]
        print("[Warn] Skipping dest_balance_delta, missing:", missing)

    if all(c in X.columns for c in ["amount", "oldbalanceDest"]):
        X["amount_to_oldbalanceDest"] = (X["amount"] / (X["oldbalanceDest"] + eps)).astype(np.float32)
    else:
        missing = [c for c in ["amount", "oldbalanceDest"] if c not in X.columns]
        print("[Warn] Skipping amount_to_oldbalanceDest, missing:", missing)

    if all(c in X.columns for c in ["oldbalanceDest", "amount", "newbalanceDest"]):
        X["dest_error"] = (X["oldbalanceDest"] + X["amount"] - X["newbalanceDest"]).astype(np.float32)
    else:
        missing = [c for c in ["oldbalanceDest", "amount", "newbalanceDest"] if c not in X.columns]
        print("[Warn] Skipping dest_error, missing:", missing)

    # Clip infinity and large values to prevent model training issues
    for col in X.select_dtypes(include=[np.floating]).columns:
        max_val = np.finfo(np.float32).max / 2  # Use a safe max value
        X[col] = X[col].clip(-max_val, max_val).fillna(0.0)

    return X

# Apply safely
X_train_raw = add_safe_features(X_train_raw)
X_test_raw  = add_safe_features(X_test_raw)

print("After FE:", X_train_raw.shape, X_test_raw.shape)



[Warn] Skipping orig_balance_delta, missing: ['newbalanceOrig']
[Warn] Skipping orig_error, missing: ['newbalanceOrig']
[Warn] Skipping dest_balance_delta, missing: ['newbalanceDest']
[Warn] Skipping dest_error, missing: ['newbalanceDest']
[Warn] Skipping orig_balance_delta, missing: ['newbalanceOrig']
[Warn] Skipping orig_error, missing: ['newbalanceOrig']
[Warn] Skipping dest_balance_delta, missing: ['newbalanceDest']
[Warn] Skipping dest_error, missing: ['newbalanceDest']
After FE: (5090094, 7) (1272526, 7)


In [20]:
# Ensure float32 for numeric columns
for c in X_train_raw.select_dtypes(include=[np.number]).columns:
    X_train_raw[c] = X_train_raw[c].astype(np.float32)
    X_test_raw[c]  = X_test_raw[c].astype(np.float32)

num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train_raw.columns if c not in num_cols]

numeric_pipe = Pipeline([
    # No median imputer (heavy). Most PaySim numeric cols have no NaNs.
    # If you have NaNs, use constant:
    ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
    ("scaler", StandardScaler(with_mean=False))  # with_mean=False avoids densifying sparse output
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

print("Numeric cols:", len(num_cols), "| Cat cols:", len(cat_cols), cat_cols)


Numeric cols: 6 | Cat cols: 1 ['type']


In [21]:
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV

def safe_predict_proba(pipe, X):
    """Return proba if supported; otherwise convert decision_function to calibrated-ish probabilities."""
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)
    if hasattr(pipe, "decision_function"):
        scores = pipe.decision_function(X)
        # binary
        if scores.ndim == 1:
            p1 = 1.0 / (1.0 + np.exp(-scores))
            return np.vstack([1.0 - p1, p1]).T.astype(np.float32)
        # multiclass softmax
        exps = np.exp(scores - scores.max(axis=1, keepdims=True))
        return (exps / exps.sum(axis=1, keepdims=True)).astype(np.float32)
    # fallback (not great)
    y_pred = pipe.predict(X)
    proba = np.zeros((len(y_pred), 2), dtype=np.float32)
    proba[:, 1] = (y_pred == 1).astype(np.float32)
    proba[:, 0] = 1.0 - proba[:, 1]
    return proba

def compute_metrics(y_true, y_pred, proba):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    m = {}
    m["acc"] = float(accuracy_score(y_true, y_pred))
    m["f1_w"] = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    m["f1_m"] = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    m["bal_acc"] = float(balanced_accuracy_score(y_true, y_pred))
    m["mcc"] = float(matthews_corrcoef(y_true, y_pred))
    m["kappa"] = float(cohen_kappa_score(y_true, y_pred))

    # AUC metrics need probability for positive class
    try:
        p1 = proba[:, 1]
        m["roc_auc_ovr_macro"] = float(roc_auc_score(y_true, p1))
        m["pr_auc_ovr_macro"] = float(average_precision_score(y_true, p1))
        m["logloss"] = float(log_loss(y_true, proba, labels=[0, 1]))
    except Exception:
        m["roc_auc_ovr_macro"] = float("nan")
        m["pr_auc_ovr_macro"] = float("nan")
        m["logloss"] = float("nan")
    return m

# ------------------------------------------------------------
# Models: fast + class-balanced + sparse-friendly
# ------------------------------------------------------------
models = {
    "LogReg_saga": LogisticRegression(
        solver="saga",
        max_iter=2000,
        n_jobs=-1,
        class_weight="balanced",
        random_state=SEED
    ),
    "SGD_logloss": SGDClassifier(
        loss="log_loss",
        alpha=1e-5,
        max_iter=2000,
        tol=1e-3,
        class_weight="balanced",
        random_state=SEED
    ),
    # RidgeClassifier has no predict_proba; we calibrate it to get proba + logloss/AUC
    "Ridge_calibrated": CalibratedClassifierCV(
        estimator=RidgeClassifier(class_weight="balanced", random_state=SEED),
        method="sigmoid",
        cv=3
    )
}

base_results = {"models": {}}
ml_results = []

# ------------------------------------------------------------
# Train + Evaluate
# ------------------------------------------------------------
for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    t0 = time.perf_counter()
    pipe.fit(X_train_raw, y_train)
    train_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    y_pred = pipe.predict(X_test_raw)
    pred_time = time.perf_counter() - t1

    proba = safe_predict_proba(pipe, X_test_raw)
    metrics = compute_metrics(y_test, y_pred, proba)

    # ---- Save EXACTLY in your structure
    base_results["models"][name] = {
        "pipeline": pipe,
        "train_time_s": float(train_time),
        "pred_time_s": float(pred_time),
        "y_true": np.asarray(y_test),
        "y_pred": np.asarray(y_pred),
        "proba": np.asarray(proba, dtype=np.float32),
        "metrics": metrics,
    }

    print(
        f"{name} | Acc={metrics['acc']:.4f} | F1(w)={metrics['f1_w']:.4f} | "
        f"F1(m)={metrics['f1_m']:.4f} | BalAcc={metrics['bal_acc']:.4f} | "
        f"MCC={metrics['mcc']:.4f} | ROC_AUC={metrics['roc_auc_ovr_macro']:.4f} | "
        f"PR_AUC={metrics['pr_auc_ovr_macro']:.4f} | "
        f"Train={train_time:.2f}s | Pred={pred_time:.2f}s"
    )

    ml_results.append({
        "Model": name,
        "Accuracy": metrics["acc"],
        "F1_weighted": metrics["f1_w"],
        "F1_macro": metrics["f1_m"],
        "BalancedAcc": metrics["bal_acc"],
        "MCC": metrics["mcc"],
        "Kappa": metrics["kappa"],
        "ROC_AUC_macroOVR": metrics["roc_auc_ovr_macro"],
        "PR_AUC_macro": metrics["pr_auc_ovr_macro"],
        "LogLoss": metrics["logloss"],
        "TrainTime_s": float(train_time),
        "PredTime_s": float(pred_time),
    })

results_df = pd.DataFrame(ml_results).sort_values(by="PR_AUC_macro", ascending=False)
results_df

c:\Users\shahi\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogReg_saga | Acc=0.9987 | F1(w)=0.9981 | F1(m)=0.4997 | BalAcc=0.5000 | MCC=0.0000 | ROC_AUC=0.5000 | PR_AUC=0.0013 | Train=6597.49s | Pred=0.70s
SGD_logloss | Acc=0.9987 | F1(w)=0.9981 | F1(m)=0.4997 | BalAcc=0.5000 | MCC=0.0000 | ROC_AUC=0.5000 | PR_AUC=0.0013 | Train=61.98s | Pred=0.60s
Ridge_calibrated | Acc=0.9987 | F1(w)=0.9981 | F1(m)=0.4997 | BalAcc=0.5000 | MCC=0.0000 | ROC_AUC=0.5000 | PR_AUC=0.0013 | Train=16.15s | Pred=0.87s


,Model,Accuracy,F1_weighted,F1_macro,BalancedAcc,MCC,Kappa,ROC_AUC_macroOVR,PR_AUC_macro,LogLoss,TrainTime_s,PredTime_s
0,LogReg_saga,0.998708,0.998063,0.499677,0.5,0.0,0.0,0.5,0.001292,0.046565,6597.493931,0.702724
1,SGD_logloss,0.998708,0.998063,0.499677,0.5,0.0,0.0,0.5,0.001292,0.046565,61.978882,0.602905
2,Ridge_calibrated,0.998708,0.998063,0.499677,0.5,0.0,0.0,0.5,0.001292,0.046565,16.152562,0.873445
